In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
print(env)
brznTablName = f"saleslake_{env}.bronze_{env}.rawCustomer"
# print(brznTablName)
srcFileLoc=f"/Volumes/saleslake_{env}/bronze_{env}/vol_saleslake_src_files_{env}/daily_customer/"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS saleslake_{env}.bronze_{env}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {brznTablName} (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    city STRING,
    state STRING,
    country STRING,
    zip_code STRING,
    segment STRING,
    ingest_ts TIMESTAMP
)
""")


In [0]:
slvrTablName = f"saleslake_{env}.silver_{env}.cleanedCustomer"

spark.sql(f"""
CREATE OR REPLACE TABLE {slvrTablName}
(
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    city STRING,
    state STRING,
    country STRING,
    zip_code STRING,
    segment STRING,
    ingest_ts TIMESTAMP
)
""")

In [0]:
goldTablName = f"saleslake_{env}.gold_{env}.refinedcustomer"

spark.sql(f"""
CREATE OR REPLACE TABLE {goldTablName}
(
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    city STRING,
    state STRING,
    country STRING,
    zip_code STRING,
    segment STRING,
    ingest_ts TIMESTAMP,
    is_active STRING,
    rec_version INTEGER,
    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP
)
""")

In [0]:

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_{env}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawProduct
(
    product_id STRING,
    sku STRING,
    product_name STRING,
    category STRING,
    sub_category STRING,
    brand STRING,
    supplier STRING,
    unit_cost STRING,
    list_price STRING,
    launch_date STRING,
    status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawStore
(
    store_id STRING,
    store_code STRING,
    store_name STRING,
    store_type STRING,
    address STRING,
    city STRING,
    state STRING,
    region_id STRING,
    manager_name STRING,
    opening_date STRING,
    square_feet STRING,
    status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawRegion
(
    region_id STRING,
    region_code STRING,
    region_name STRING,
    country STRING,
    sub_region STRING,
    regional_manager STRING,
    created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawDiscount
(
    discount_id STRING,
    discount_code STRING,
    discount_name STRING,
    discount_type STRING,
    discount_value STRING,
    min_amount STRING,
    max_amount STRING,
    valid_from STRING,
    valid_to STRING,
    applicable_segment STRING,
    applicable_category STRING,
    usage_limit STRING,
    is_active STRING,
    created_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawSales
(
    sale_id STRING,
    product_id STRING,
    store_id STRING,
    customer_id STRING,
    region_id STRING,
    quantity STRING,
    unit_price STRING,
    gross_amount STRING,
    discount_code STRING,
    discount_amount STRING,
    net_amount STRING,
    cost_amount STRING,
    sale_date STRING,
    payment_method STRING,
    channel STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{bronzeSchema}.rawInvoice
(
    invoice_id STRING,
    invoice_number STRING,
    customer_id STRING,
    invoice_date STRING,
    due_date STRING,
    subtotal_amount STRING,
    discount_code STRING,
    discount_amount STRING,
    tax_amount STRING,
    total_amount STRING,
    payment_status STRING,
    payment_method STRING,
    payment_date STRING,
    currency STRING,
    region STRING,
    store_id STRING,
    channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

In [0]:
silverSchema = f"silver_{env}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedProduct
(
    product_id INT,
    sku STRING,
    product_name STRING,
    category STRING,
    sub_category STRING,
    brand STRING,
    supplier STRING,
    unit_cost DECIMAL(18,2),
    list_price DECIMAL(18,2),
    launch_date DATE,
    status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedStore
(
    store_id INT,
    store_code STRING,
    store_name STRING,
    store_type STRING,
    address STRING,
    city STRING,
    state STRING,
    region_id INT,
    manager_name STRING,
    opening_date DATE,
    square_feet INT,
    status STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedRegion
(
    region_id INT,
    region_code STRING,
    region_name STRING,
    country STRING,
    sub_region STRING,
    regional_manager STRING,
    created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedDiscount
(
    discount_id INT,
    discount_code STRING,
    discount_name STRING,
    discount_type STRING,
    discount_value DECIMAL(10,2),
    min_amount DECIMAL(18,2),
    max_amount DECIMAL(18,2),
    valid_from DATE,
    valid_to DATE,
    applicable_segment STRING,
    applicable_category STRING,
    usage_limit INT,
    is_active STRING,
    created_date DATE,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedSales
(
    sale_id BIGINT,
    product_id INT,
    store_id INT,
    customer_id INT,
    region_id INT,
    quantity INT,
    unit_price DECIMAL(18,2),
    gross_amount DECIMAL(18,2),
    discount_code STRING,
    discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2),
    cost_amount DECIMAL(18,2),
    sale_date DATE,
    payment_method STRING,
    channel STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS 
{catalog}.{silverSchema}.cleanedInvoice
(
    invoice_id BIGINT,
    invoice_number STRING,
    customer_id INT,
    invoice_date DATE,
    due_date DATE,
    subtotal_amount DECIMAL(18,2),
    discount_code STRING,
    discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING,
    payment_method STRING,
    payment_date DATE,
    currency STRING,
    region STRING,
    store_id INT,
    channel STRING,
    created_by STRING,
    ingest_ts TIMESTAMP
)
USING DELTA
""")

In [0]:
goldSchema = f"gold_{env}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.dim_date
(
    date_sk INT,
    full_date DATE,

    day_of_month INT,
    day_name STRING,
    week_of_year INT,

    month_number INT,
    month_name STRING,

    quarter INT,
    year INT,

    is_weekend BOOLEAN,
    is_month_end BOOLEAN,

    fiscal_year INT,
    fiscal_quarter STRING,

    created_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.dim_region
(
    region_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    region_id INT,
    region_code STRING,
    region_name STRING,
    country STRING,
    sub_region STRING,
    regional_manager STRING,

    initial_load_ts TIMESTAMP,
    last_update_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.dim_product
(
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    product_id INT,
    sku STRING,
    product_name STRING,
    category STRING,
    sub_category STRING,
    brand STRING,
    supplier STRING,

    unit_cost DECIMAL(18,2),
    list_price DECIMAL(18,2),
    launch_date DATE,
    status STRING,

    is_active STRING,
    rec_version INT,

    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP,

    initial_load_ts TIMESTAMP,
    last_update_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.dim_store
(
    store_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    store_id INT,
    store_code STRING,
    store_name STRING,
    store_type STRING,

    address STRING,
    city STRING,
    state STRING,

    region_id INT,
    manager_name STRING,

    opening_date DATE,
    square_feet INT,

    status STRING,

    is_active STRING,
    rec_version INT,

    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP,

    initial_load_ts TIMESTAMP,
    last_update_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.dim_discount
(
    discount_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    discount_id INT,
    discount_code STRING,
    discount_name STRING,
    discount_type STRING,

    discount_value DECIMAL(10,2),
    min_amount DECIMAL(18,2),
    max_amount DECIMAL(18,2),

    valid_from DATE,
    valid_to DATE,

    applicable_segment STRING,
    applicable_category STRING,

    usage_limit INT,
    status STRING,

    is_active STRING,
    rec_version INT,

    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP,

    initial_load_ts TIMESTAMP,
    last_update_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.fact_sales
(
    sale_id BIGINT,

    date_sk INT,
    customer_sk BIGINT,
    product_sk BIGINT,
    store_sk BIGINT,
    region_sk BIGINT,
    discount_sk BIGINT,

    quantity INT,

    unit_price DECIMAL(18,2),
    gross_amount DECIMAL(18,2),
    discount_amount DECIMAL(18,2),
    net_amount DECIMAL(18,2),
    cost_amount DECIMAL(18,2),

    profit_amount DECIMAL(18,2),
    profit_margin_pct DECIMAL(10,4),

    payment_method STRING,
    channel STRING,

    sale_date DATE,

    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (sale_date)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog}.{goldSchema}.fact_invoice
(
    invoice_id BIGINT,
    invoice_number STRING,

    customer_sk BIGINT,
    store_sk BIGINT,
    region_sk BIGINT,
    discount_sk BIGINT,

    invoice_date_sk INT,
    due_date_sk INT,
    payment_date_sk INT,

    subtotal_amount DECIMAL(18,2),
    discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),

    days_to_pay INT,

    payment_status STRING,
    is_overdue STRING,

    invoice_date DATE,

    load_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (invoice_date)
""")